In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("dataset/movie1.csv")
# print(df.head(1))
df = df[["title","plot","year"]].copy()
print(df.head(1))

In [ ]:
print(df.shape)
df_usable = df.dropna()
print(df_usable.shape)
print(df_usable.head(1))


In [ ]:
import re

def clean_plot(text):
    if not isinstance(text, str):
        return text
    # Remove non-ascii characters
    text = text.encode('ascii', 'ignore').decode()
    # Remove non-standard punctuation
    text = re.sub(r'[^a-zA-Z0-9\s.,!?;:\'"()-]', '', text)
    # Remove newlines and paragraph spaces
    text = re.sub(r'[\n\r\t]+', ' ', text)
    # Remove extra white spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_usable.loc[:, 'plot'] = df_usable['plot'].apply(clean_plot)
print("Sample cleaned plot:", df_usable['plot'].iloc[0])


In [ ]:
# Uncomment below to install NLTK & tqdm if not installed
# %pip install nltk tqdm

import nltk
from tqdm.auto import tqdm
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer

# Download the POS tagger needed to identify Nouns/Verbs
try:
    nltk.download('averaged_perceptron_tagger_eng', quiet=True) 
    nltk.download('averaged_perceptron_tagger', quiet=True)
except:
    pass

# Ensure we use the GPU by explicit device='cuda' 
# (Though SentenceTransformer auto-detects Colab GPUs)
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cuda')
kw_model = KeyBERT(model=embedder)

print(f"Starting extraction for {len(df_usable)} records on GPU...")
plot_list = df_usable['plot'].tolist()

# 1. BATCHED EXTRACTION WITH PROGRESS BAR
# We process in large chunks so we can update the progress bar without slowing down the GPU
batch_size = 1000 
raw_candidates = []

for i in tqdm(range(0, len(plot_list), batch_size), desc="KeyBERT GPU Extraction"):
    batch = plot_list[i : i + batch_size]
    batch_ext = kw_model.extract_keywords(
        batch, 
        keyphrase_ngram_range=(1, 1), 
        stop_words='english', 
        top_n=20, 
        use_mmr=True, 
        diversity=0.7
    )
    raw_candidates.extend(batch_ext)

# 2. Extract just the words for NLTK
list_of_words = [[kw[0] for kw in doc_kws] for doc_kws in raw_candidates]

# 3. BATCHED NLTK TAGGING
pos_tags_list = nltk.pos_tag_sents(list_of_words)

# 4. Filter the tags
final_keywords_list = []
for tags in tqdm(pos_tags_list, desc="NLTK Filtering"):
    # KEEP: 'NN', 'NNS', 'JJ'
    filtered = [w for w, tag in tags if tag in ('NN', 'NNS', 'JJ')]
    final_keywords_list.append(filtered[:5])

# Assign back to dataframe
df_usable['plot_keywords'] = final_keywords_list
df_usable['plot_keywords_str'] = df_usable['plot_keywords'].apply(lambda x: ', '.join(x))

print("\nExtraction Complete! Sample output:")
print(df_usable[['plot', 'plot_keywords_str']].head())


In [ ]:
df_usable["metadata_for_embedding"] = df_usable["plot_keywords_str"].astype(str)+"."+df_usable["plot"].astype(str)
# print(df_usable.head(1))
print(df_usable["metadata_for_embedding"].iloc[0])

In [ ]:

#Here add your Hugging face token
# The rest of your code:
from sentence_transformers import SentenceTransformer
# model = SentenceTransformer('all-MiniLM-L6-v2') 


In [ ]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
metadata_list =df_usable["plot"].tolist()
embeddings = model.encode(metadata_list,show_progress_bar=True)